# Hyperparameter Tuning & Model Selection

## Objective

The objective of this notebook is to optimize the baseline models identified in Notebook 3 through hyperparameter tuning.

The selected models are:

- Logistic Regression
- Support Vector Machine
- XGBoost

Each model will be tuned using cross-validation and evaluated on the same hold-out test dataset. The final production model will be selected based on predictive performance and business requirements.

In [26]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

# Model Selection
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV,
    RandomizedSearchCV
)

# Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

# Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier

In [2]:
# =============================================================================
# Load Dataset
# =============================================================================

DATA_PATH = Path("../dataset/employee_attrition.csv")

df = pd.read_csv(DATA_PATH)

print(f"Dataset Shape: {df.shape}")

df.head()

Dataset Shape: (1470, 35)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [3]:
# =============================================================================
# Remove Constant and Identifier Columns
# =============================================================================

columns_to_drop = [
    "EmployeeCount",
    "EmployeeNumber",
    "Over18",
    "StandardHours"
]

df = df.drop(columns=columns_to_drop)

print(df.shape)

(1470, 31)


In [4]:
# =============================================================================
# Separate Features and Target
# =============================================================================

X = df.drop(columns="Attrition")

y = df["Attrition"].map({
    "No": 0,
    "Yes": 1
})

print(X.shape)
print(y.shape)

(1470, 30)
(1470,)


In [5]:
# =============================================================================
# Train-Test Split
# =============================================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

Training Shape: (1176, 30)
Testing Shape: (294, 30)


In [6]:
# =============================================================================
# Numerical and Categorical Features
# =============================================================================

numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include="object"
).columns.tolist()

print(f"Numerical Features: {len(numerical_features)}")
print(f"Categorical Features: {len(categorical_features)}")

Numerical Features: 23
Categorical Features: 7


In [7]:
# =============================================================================
# Numerical Pipeline
# =============================================================================

numerical_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

# =============================================================================
# Categorical Pipeline
# =============================================================================

categorical_pipeline = Pipeline(
    steps=[
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

# =============================================================================
# Column Transformer
# =============================================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            numerical_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

In [8]:
# =============================================================================
# Transform Dataset
# =============================================================================

X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print(X_train_processed.shape)
print(X_test_processed.shape)

(1176, 51)
(294, 51)


In [9]:
# =============================================================================
# Cross-Validation Strategy
# =============================================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print(cv)

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)


In [10]:
# =============================================================================
# Logistic Regression Parameter Grid
# =============================================================================

lr_param_grid = {

    "C": [0.01, 0.1, 1, 10, 100],

    "penalty": ["l1", "l2"],

    "solver": ["liblinear"],

    "class_weight": [
        None,
        "balanced"
    ]

}

In [11]:
# =============================================================================
# Logistic Regression Grid Search
# =============================================================================

lr_grid = GridSearchCV(

    estimator=LogisticRegression(

        max_iter=1000,

        random_state=42

    ),

    param_grid=lr_param_grid,

    scoring="f1",

    cv=cv,

    n_jobs=-1,

    verbose=1

)

In [12]:
# =============================================================================
# Train Grid Search
# =============================================================================

lr_grid.fit(
    X_train_processed,
    y_train
)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LogisticRegre...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': [0.01, 0.1, ...], 'class_weight': [None, 'balanced'], 'penalty': ['l1', 'l2'], 'solver': ['liblinear']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_

In [36]:
print("Best Parameters:")

lr_grid.best_params_

Best Parameters:


{'C': 1, 'class_weight': None, 'penalty': 'l1', 'solver': 'liblinear'}

In [13]:
print("Best F1 Score")

lr_grid.best_score_

Best F1 Score


np.float64(0.564025358639902)

In [14]:
best_lr = lr_grid.best_estimator_

In [15]:
# =============================================================================
# Support Vector Machine Parameter Grid
# =============================================================================

svm_param_grid = {

    "C": [0.1, 1, 10, 100],

    "kernel": [
        "linear",
        "rbf"
    ],

    "gamma": [
        "scale",
        "auto"
    ],

    "class_weight": [
        None,
        "balanced"
    ]

}

In [16]:
# =============================================================================
# SVM Grid Search
# =============================================================================

svm_grid = GridSearchCV(

    estimator=SVC(
        probability=True,
        random_state=42
    ),

    param_grid=svm_param_grid,

    scoring="f1",

    cv=cv,

    n_jobs=-1,

    verbose=1

)

In [17]:
# =============================================================================
# Train Grid Search
# =============================================================================

svm_grid.fit(
    X_train_processed,
    y_train
)

Fitting 5 folds for each of 32 candidates, totalling 160 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",SVC(probabili...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': [0.1, 1, ...], 'class_weight': [None, 'balanced'], 'gamma': ['scale', 'auto'], 'kernel': ['linear', 'rbf']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`s

In [18]:
print("Best Parameters:")

svm_grid.best_params_

Best Parameters:


{'C': 10, 'class_weight': None, 'gamma': 'scale', 'kernel': 'linear'}

In [19]:
print("Best CV F1 Score:")

svm_grid.best_score_

Best CV F1 Score:


np.float64(0.5719490158320596)

In [20]:
best_svm = svm_grid.best_estimator_

In [21]:
# =============================================================================
# XGBoost Parameter Distribution
# =============================================================================

xgb_param_dist = {

    "n_estimators": [100, 200, 300],

    "max_depth": [3, 5, 7, 9],

    "learning_rate": [0.01, 0.05, 0.1, 0.2],

    "subsample": [0.8, 0.9, 1.0],

    "colsample_bytree": [0.8, 0.9, 1.0],

    "min_child_weight": [1, 3, 5]

}

In [27]:
# =============================================================================
# XGBoost Random Search
# =============================================================================

xgb_random = RandomizedSearchCV(

    estimator=XGBClassifier(

        objective="binary:logistic",

        eval_metric="logloss",

        random_state=42,

        use_label_encoder=False

    ),

    param_distributions=xgb_param_dist,

    n_iter=30,

    scoring="f1",

    cv=cv,

    random_state=42,

    n_jobs=-1,

    verbose=1

)

In [28]:
# =============================================================================
# Train Random Search
# =============================================================================

xgb_random.fit(
    X_train_processed,
    y_train
)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBClassifier...ree=None, ...)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'colsample_bytree': [0.8, 0.9, ...], 'learning_rate': [0.01, 0.05, ...], 'max_depth': [3, 5, ...], 'min_child_weight': [1, 3, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",30
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` c

In [29]:
print("Best Parameters")

xgb_random.best_params_

Best Parameters


{'subsample': 0.8,
 'n_estimators': 200,
 'min_child_weight': 3,
 'max_depth': 5,
 'learning_rate': 0.2,
 'colsample_bytree': 0.8}

In [30]:
print("Best CV F1 Score")

xgb_random.best_score_

Best CV F1 Score


np.float64(0.557319831826191)

In [31]:
best_xgb = xgb_random.best_estimator_

In [32]:
def evaluate_model(model, X_test, y_test):

    y_pred = model.predict(X_test)

    y_prob = model.predict_proba(X_test)[:, 1]

    results = {

        "Accuracy": accuracy_score(y_test, y_pred),

        "Precision": precision_score(y_test, y_pred),

        "Recall": recall_score(y_test, y_pred),

        "F1 Score": f1_score(y_test, y_pred),

        "ROC AUC": roc_auc_score(y_test, y_prob)

    }

    return results

In [33]:
lr_results = evaluate_model(
    best_lr,
    X_test_processed,
    y_test
)

svm_results = evaluate_model(
    best_svm,
    X_test_processed,
    y_test
)

xgb_results = evaluate_model(
    best_xgb,
    X_test_processed,
    y_test
)

In [34]:
comparison = pd.DataFrame({

    "Logistic Regression": lr_results,

    "Support Vector Machine": svm_results,

    "XGBoost": xgb_results

}).T

comparison

,Accuracy,Precision,Recall,F1 Score,ROC AUC
Logistic Regression,0.863946,0.629630,0.361702,0.459459,0.809114
Support Vector Machine,0.867347,0.666667,0.340426,0.450704,0.813378
XGBoost,0.853741,0.590909,0.276596,0.376812,0.779998


## Conclusion

Three shortlisted models—Logistic Regression, Support Vector Machine (SVM), and XGBoost—were optimized using hyperparameter tuning with cross-validation.

Among the tuned models, Logistic Regression achieved the highest F1-score (0.4595) and Recall (0.3617), making it the most balanced model for predicting employee attrition. Support Vector Machine showed the greatest improvement over its baseline and achieved the highest Accuracy, Precision, and ROC-AUC, but its Recall remained slightly lower than Logistic Regression. XGBoost did not improve after tuning and underperformed its baseline model.

Considering both the quantitative evaluation metrics and the business objective of identifying employees at risk of leaving, Logistic Regression was selected as the final model for the production pipeline.